# 04 — Patna Case Study: Boring Road & Kurji Area

Apply trained models to detect urban change in Patna, Bihar.

**Areas of interest**:
- **Boring Road** (~25.61°N, 85.105°E): Commercial corridor, dense urban development
- **Kurji** (~25.585°N, 85.09°E): Residential expansion, institutional area

In [ ]:
!pip install pystac-client planetary-computer rasterio -q

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

if os.path.exists('delta/src'):
    sys.path.insert(0, 'delta')
elif os.path.exists('../src'):
    sys.path.insert(0, '..')

import torch
from src.inference.predict import load_model, predict, predict_autoencoder, preprocess_pair
from src.data.ndvi import compute_ndvi
from src.inference.visualize import plot_triptych, plot_change_heatmap, plot_ndvi_comparison

print("Imports OK")

## 1. Search Sentinel-2 Imagery via Planetary Computer

In [ ]:
import pystac_client
import planetary_computer

PATNA_AOI = [85.08, 25.57, 85.12, 25.62]  # [west, south, east, north]

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

print("Connected to Microsoft Planetary Computer")

In [ ]:
search_t1 = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=PATNA_AOI,
    datetime="2022-01-01/2022-12-31",
    query={"eo:cloud_cover": {"lt": 10}},
    limit=10,
    sortby=[{"field": "eo:cloud_cover", "direction": "asc"}],
)
t1_items = list(search_t1.items())
print(f"T1 scenes found: {len(t1_items)}")

for item in t1_items[:3]:
    cloud = item.properties.get('eo:cloud_cover', 'N/A')
    date = item.datetime.strftime('%Y-%m-%d')
    print(f"  {date} — Cloud: {cloud:.1f}%")

In [ ]:
search_t2 = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=PATNA_AOI,
    datetime="2024-01-01/2024-12-31",
    query={"eo:cloud_cover": {"lt": 10}},
    limit=10,
    sortby=[{"field": "eo:cloud_cover", "direction": "asc"}],
)
t2_items = list(search_t2.items())
print(f"T2 scenes found: {len(t2_items)}")

for item in t2_items[:3]:
    cloud = item.properties.get('eo:cloud_cover', 'N/A')
    date = item.datetime.strftime('%Y-%m-%d')
    print(f"  {date} — Cloud: {cloud:.1f}%")

## 2. Download and Load Bands

In [ ]:
import rasterio
from rasterio.windows import from_bounds
from rasterio.transform import from_bounds as make_transform

def load_band(item, band_name, bbox, target_shape=(256, 256)):
    href = item.assets[band_name].href
    with rasterio.open(href) as src:
        window = from_bounds(bbox[0], bbox[1], bbox[2], bbox[3], src.transform)
        data = src.read(1, window=window).astype(np.float32)
    data = np.array(Image.fromarray(data).resize(target_shape, Image.BILINEAR))
    return data

if len(t1_items) > 0 and len(t2_items) > 0:
    best_t1 = t1_items[0]
    best_t2 = t2_items[0]
    print(f"T1: {best_t1.datetime.strftime('%Y-%m-%d')} (cloud: {best_t1.properties['eo:cloud_cover']:.1f}%)")
    print(f"T2: {best_t2.datetime.strftime('%Y-%m-%d')} (cloud: {best_t2.properties['eo:cloud_cover']:.1f}%)")
else:
    print("No scenes found - try wider date range or higher cloud tolerance")
    best_t1 = best_t2 = None

In [ ]:
if best_t1 and best_t2:
    bands_config = ['B04', 'B03', 'B02', 'B08']  # R, G, B, NIR
    band_map = {'B04': 'red', 'B03': 'green', 'B02': 'blue', 'B08': 'nir'}

    t1_bands = {}
    t2_bands = {}

    for band in bands_config:
        print(f"Loading {band} ({band_map[band]})...")
        t1_bands[band] = load_band(best_t1, band, PATNA_AOI)
        t2_bands[band] = load_band(best_t2, band, PATNA_AOI)

    t1_rgb = np.stack([t1_bands['B04'], t1_bands['B03'], t1_bands['B02']], axis=-1)
    t2_rgb = np.stack([t2_bands['B04'], t2_bands['B03'], t2_bands['B02']], axis=-1)

    t1_rgb_vis = np.clip(t1_rgb / 3000, 0, 1)
    t2_rgb_vis = np.clip(t2_rgb / 3000, 0, 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(t1_rgb_vis)
    axes[0].set_title(f'T1: {best_t1.datetime.strftime("%Y-%m-%d")}')
    axes[0].axis('off')
    axes[1].imshow(t2_rgb_vis)
    axes[1].set_title(f'T2: {best_t2.datetime.strftime("%Y-%m-%d")}')
    axes[1].axis('off')
    plt.suptitle('Patna Sentinel-2 imagery', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 3. Run Supervised Model

In [ ]:
def prepare_model_input(t1_bands, t2_bands):
    bands = ['B04', 'B03', 'B02', 'B08']
    t1 = np.stack([t1_bands[b] for b in bands], axis=0).astype(np.float32) / 10000.0
    t2 = np.stack([t2_bands[b] for b in bands], axis=0).astype(np.float32) / 10000.0

    t1_tensor = torch.from_numpy(t1).unsqueeze(0)
    t2_tensor = torch.from_numpy(t2).unsqueeze(0)

    nir_t1 = t1[3]
    red_t1 = t1[0]
    ndvi_t1 = compute_ndvi(nir_t1, red_t1)
    t1_tensor = torch.cat([t1_tensor, torch.from_numpy(ndvi_t1).unsqueeze(0).unsqueeze(0)], dim=1)

    nir_t2 = t2[3]
    red_t2 = t2[0]
    ndvi_t2 = compute_ndvi(nir_t2, red_t2)
    t2_tensor = torch.cat([t2_tensor, torch.from_numpy(ndvi_t2).unsqueeze(0).unsqueeze(0)], dim=1)

    return t1_tensor, t2_tensor

print("Preprocessing function ready")

In [ ]:
if best_t1 and best_t2:
    t1_tensor, t2_tensor = prepare_model_input(t1_bands, t2_bands)
    print(f"T1 tensor: {t1_tensor.shape}")
    print(f"T2 tensor: {t2_tensor.shape}")

    checkpoint_sup = 'siamese_unet_best.pt'
    if os.path.exists(checkpoint_sup):
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model_sup, config_sup = load_model(checkpoint_sup, device)

        with torch.no_grad():
            stacked = torch.cat([t1_tensor, t2_tensor], dim=1).to(device)
            pred_sup = torch.sigmoid(model_sup(stacked)).squeeze().cpu().numpy()

        pred_binary_sup = (pred_sup > 0.5).astype(np.uint8)
        print(f"Supervised prediction: {pred_binary_sup.shape}")
        print(f"Changed pixels: {pred_binary_sup.sum()} / {pred_binary_sup.size} ({pred_binary_sup.mean()*100:.1f}%)")
    else:
        print(f"Checkpoint not found: {checkpoint_sup}")
        print("Upload siamese_unet_best.pt from Colab training (notebook 02)")
        pred_binary_sup = np.zeros((256, 256), dtype=np.uint8)
        pred_sup = np.zeros((256, 256), dtype=np.float32)

## 4. Run Self-Supervised Model

In [ ]:
if best_t1 and best_t2:
    checkpoint_ss = 'autoencoder_best.pt'
    if os.path.exists(checkpoint_ss):
        model_ss, config_ss = load_model(checkpoint_ss, device)

        with torch.no_grad():
            t1_dev = t1_tensor.to(device)
            t2_dev = t2_tensor.to(device)
            recon_t1 = model_ss(t1_dev)
            recon_t2 = model_ss(t2_dev)

        err_t1 = torch.mean((t1_dev - recon_t1)**2, dim=1).squeeze().cpu().numpy()
        err_t2 = torch.mean((t2_dev - recon_t2)**2, dim=1).squeeze().cpu().numpy()
        change_map_ss = np.abs(err_t1 - err_t2)

        pred_binary_ss = (change_map_ss > 0.15).astype(np.uint8)
        print(f"Self-supervised prediction: {pred_binary_ss.shape}")
        print(f"Changed pixels: {pred_binary_ss.sum()} / {pred_binary_ss.size} ({pred_binary_ss.mean()*100:.1f}%)")
    else:
        print(f"Checkpoint not found: {checkpoint_ss}")
        print("Upload autoencoder_best.pt from Colab training (notebook 03)")
        pred_binary_ss = np.zeros((256, 256), dtype=np.uint8)
        change_map_ss = np.zeros((256, 256), dtype=np.float32)

## 5. NDVI Difference (Baseline)

In [ ]:
if best_t1 and best_t2:
    ndvi_t1 = compute_ndvi(t1_bands['B08'], t1_bands['B04'])
    ndvi_t2 = compute_ndvi(t2_bands['B08'], t2_bands['B04'])
    ndvi_diff = ndvi_t2 - ndvi_t1

    ndvi_binary = (np.abs(ndvi_diff) > 0.2).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    im1 = axes[0].imshow(ndvi_t1, cmap='RdYlGn', vmin=-1, vmax=1)
    axes[0].set_title('NDVI Before')
    axes[0].axis('off')
    plt.colorbar(im1, ax=axes[0], fraction=0.046)

    im2 = axes[1].imshow(ndvi_t2, cmap='RdYlGn', vmin=-1, vmax=1)
    axes[1].set_title('NDVI After')
    axes[1].axis('off')
    plt.colorbar(im2, ax=axes[1], fraction=0.046)

    im3 = axes[2].imshow(ndvi_diff, cmap='RdYlGn', vmin=-0.5, vmax=0.5)
    axes[2].set_title('NDVI Change')
    axes[2].axis('off')
    plt.colorbar(im3, ax=axes[2], fraction=0.046)

    plt.suptitle('NDVI Change Analysis - Patna', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('patna_ndvi_change.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Side-by-Side Comparison

In [ ]:
if best_t1 and best_t2:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    axes[0, 0].imshow(t1_rgb_vis)
    axes[0, 0].set_title(f'Before ({best_t1.datetime.strftime("%Y-%m-%d")})')
    axes[0, 0].axis('off')

    axes[0, 1].imshow(t2_rgb_vis)
    axes[0, 1].set_title(f'After ({best_t2.datetime.strftime("%Y-%m-%d")})')
    axes[0, 1].axis('off')

    axes[0, 2].imshow(t2_rgb_vis, alpha=0.6)
    change_rgb = np.zeros((*pred_binary_sup.shape, 3))
    change_rgb[pred_binary_sup > 0] = [1, 0, 0]
    axes[0, 2].imshow(change_rgb, alpha=0.5)
    pct = pred_binary_sup.mean() * 100
    axes[0, 2].set_title(f'Supervised ({pct:.1f}% changed)')
    axes[0, 2].axis('off')

    axes[1, 0].imshow(t2_rgb_vis, alpha=0.6)
    change_ss = np.zeros((*pred_binary_ss.shape, 3))
    change_ss[pred_binary_ss > 0] = [0, 0, 1]
    axes[1, 0].imshow(change_ss, alpha=0.5)
    pct_ss = pred_binary_ss.mean() * 100
    axes[1, 0].set_title(f'Self-Supervised ({pct_ss:.1f}% changed)')
    axes[1, 0].axis('off')

    axes[1, 1].imshow(change_map_ss, cmap='hot')
    axes[1, 1].set_title('Reconstruction Error Map')
    axes[1, 1].axis('off')

    ndvi_binary_rgb = np.zeros((*ndvi_binary.shape, 3))
    ndvi_binary_rgb[ndvi_binary > 0] = [0, 1, 0]
    axes[1, 2].imshow(t2_rgb_vis, alpha=0.6)
    axes[1, 2].imshow(ndvi_binary_rgb, alpha=0.5)
    axes[1, 2].set_title(f'NDVI Baseline ({ndvi_binary.mean()*100:.1f}% changed)')
    axes[1, 2].axis('off')

    plt.suptitle('Patna Change Detection - All Methods', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('patna_comparison_all.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Statistics

In [ ]:
pixel_size_m = 10  # Sentinel-2 resolution
hectare_per_pixel = (pixel_size_m ** 2) / 10000

methods = {
    'Supervised Siamese U-Net': pred_binary_sup,
    'Self-Supervised Autoencoder': pred_binary_ss,
    'NDVI Baseline': ndvi_binary if best_t1 else np.zeros((256, 256)),
}

print(f"{'Method':<35} {'Changed px':>12} {'Area (ha)':>12} {'% Area':>10}")
print("-" * 70)
for name, mask in methods.items():
    changed = mask.sum()
    area = changed * hectare_per_pixel
    pct = mask.mean() * 100
    print(f"{name:<35} {changed:>12,} {area:>12.2f} {pct:>9.2f}%")

## Summary

- Supervised model: trained on Texas buildings in OSCD, applied to Patna urban fabric
- Self-supervised model: no labels needed, detects any reconstruction anomaly as change
- NDVI baseline: simple vegetation index change thresholding
- **Limitation**: OSCD is ~0.5m resolution (Google Earth), Patna Sentinel-2 is 10m. Direct comparison is limited.
- Next: `05_evaluation.ipynb` for detailed metrics and visualizations